# BehaviourSpace Sensitivity Analysis

Notebook для аналізу результатів NetLogo BehaviourSpace.

Мета: зрозуміти, які параметри моделі найбільше впливають на KPI: profit, revenue, visits, loyalty cost, satisfaction тощо.

## 1. Імпорт бібліотек

In [21]:
import pandas as pd
import re
import ast
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

pd.set_option("display.max_columns", 100)

## 2. Завантаження CSV з BehaviourSpace

Вкажи шлях до файлу, який експортував NetLogo BehaviourSpace.

In [22]:
def keep_final_step_columns(df):
    base_cols = []
    metric_groups = {}

    for col in df.columns:
        match = re.match(r"^(.*)\.(\d+)$", col)

        if match:
            base_name = match.group(1)
            step = int(match.group(2))
            metric_groups.setdefault(base_name, []).append((step, col))
        else:
            base_cols.append(col)

    final_metric_cols = {}

    for base_name, items in metric_groups.items():
        last_step, last_col = max(items, key=lambda x: x[0])
        final_metric_cols[last_col] = base_name

    final_df = df[base_cols + list(final_metric_cols.keys())].copy()
    final_df = final_df.rename(columns=final_metric_cols)

    return final_df


In [23]:
file_path = "./loyalty-cafes-simple-experiment-spreadsheet.csv"
# file_path = "./loyalty-cafes-simple-experiment-stats.csv"

df_source = pd.read_csv(file_path, skiprows=23)

df = keep_final_step_columns(df_source)

df.columns.tolist()

df.head()


,[all run data],[step],sum [profit] of cafes,sum [revenue] of cafes,sum [loyalty-cost] of cafes,sum [visits] of cafes,sum [lost-visits] of customers,mean [satisfaction] of customers,[profit] of one-of cafes with [cafe-id = 0],[profit] of one-of cafes with [cafe-id = 1],[profit] of one-of cafes with [cafe-id = 2],[profit] of one-of cafes with [cafe-id = 3],[profit] of one-of cafes with [cafe-id = 4],[visits] of one-of cafes with [cafe-id = 0],[visits] of one-of cafes with [cafe-id = 1],[visits] of one-of cafes with [cafe-id = 2],[visits] of one-of cafes with [cafe-id = 3],[visits] of one-of cafes with [cafe-id = 4],[step],sum [profit] of cafes,sum [revenue] of cafes,sum [loyalty-cost] of cafes,sum [visits] of cafes,sum [lost-visits] of customers,mean [satisfaction] of customers,[profit] of one-of cafes with [cafe-id = 0],[profit] of one-of cafes with [cafe-id = 1],[profit] of one-of cafes with [cafe-id = 2],[profit] of one-of cafes with [cafe-id = 3],[profit] of one-of cafes with [cafe-id = 4],[visits] of one-of cafes with [cafe-id = 0],[visits] of one-of cafes with [cafe-id = 1],[visits] of one-of cafes with [cafe-id = 2],[visits] of one-of cafes with [cafe-id = 3],[visits] of one-of cafes with [cafe-id = 4]
0,NaN,0,0.0,0,0.0,0,0,0.751137,0,0,0,0,0.0,0,0,0,0,0,0,0.0,0,0.0,0,0,0.756207,0,0.0,0,0,0.0,0,0,0,0,0
1,NaN,1,14805.0,58000,1695.0,1700,94,0.758628,3500,2750,2750,3000,2805.0,500,500,500,100,100,1,17448.5,63390,1792.5,1763,0,0.762047,3241,2750.0,2750,4500,4207.5,463,500,500,150,150
2,NaN,2,29515.0,115935,3420.0,3400,145,0.766402,7000,5500,5500,5905,5610.0,1000,1000,1000,200,200,2,34585.0,125785,3615.0,3495,0,0.767211,6265,5500.0,5500,8905,8415.0,895,1000,1000,300,300
3,NaN,3,44130.0,173805,5175.0,5100,178,0.774089,10500,8250,8250,8715,8415.0,1500,1500,1500,300,300,3,51854.5,188750,5437.5,5246,0,0.772246,9422,8250.0,8250,13310,12622.5,1346,1500,1500,450,450
4,NaN,4,58359.8,231415,7055.2,6800,272,0.780903,14000,11000,11000,11145,11214.8,2000,2000,2000,400,400,4,68297.0,251250,7530.0,7001,0,0.777496,12607,11000.0,11000,16860,16830.0,1801,2000,2000,600,600


In [24]:
def clean_behaviorspace_final(df):
    base_cols = []
    metric_groups = {}

    for col in df.columns:
        match = re.match(r"^(.*)\.(\d+)$", str(col))

        if match:
            base_name = match.group(1)
            step = int(match.group(2))
            metric_groups.setdefault(base_name, []).append((step, col))
        else:
            base_cols.append(col)

    final_cols = {}

    for base_name, items in metric_groups.items():
        _, last_col = max(items, key=lambda x: x[0])
        final_cols[last_col] = base_name

    result = df[base_cols + list(final_cols.keys())].copy()
    result = result.rename(columns=final_cols)

    # прибрати дублікати типу [step], sum [profit] of cafes тощо
    result = result.loc[:, ~result.columns.duplicated(keep="last")]

    # прибрати службову колонку, якщо вона не потрібна
    if "[all run data]" in result.columns:
        result = result.drop(columns=["[all run data]"])

    return result

df = clean_behaviorspace_final(df)

df.columns.tolist()


['[step]',
 'sum [profit] of cafes',
 'sum [revenue] of cafes',
 'sum [loyalty-cost] of cafes',
 'sum [visits] of cafes',
 'sum [lost-visits] of customers',
 'mean [satisfaction] of customers',
 '[profit] of one-of cafes with [cafe-id = 0]',
 '[profit] of one-of cafes with [cafe-id = 1]',
 '[profit] of one-of cafes with [cafe-id = 2]',
 '[profit] of one-of cafes with [cafe-id = 3]',
 '[profit] of one-of cafes with [cafe-id = 4]',
 '[visits] of one-of cafes with [cafe-id = 0]',
 '[visits] of one-of cafes with [cafe-id = 1]',
 '[visits] of one-of cafes with [cafe-id = 2]',
 '[visits] of one-of cafes with [cafe-id = 3]',
 '[visits] of one-of cafes with [cafe-id = 4]']

In [25]:
def parse_all_run_data(df):
    if "[all run data]" not in df.columns:
        raise ValueError("Column [all run data] not found")

    parsed_rows = []

    for value in df["[all run data]"]:
        try:
            items = ast.literal_eval(value)
        except Exception:
            parsed_rows.append({})
            continue

        row = {}

        for item in items:
            if isinstance(item, list) and len(item) >= 2:
                row[item[0]] = item[1]

        parsed_rows.append(row)

    params_df = pd.DataFrame(parsed_rows)

    return pd.concat(
        [params_df, df.drop(columns=["[all run data]"])],
        axis=1
    )

df = parse_all_run_data(df)
df.head()

ValueError: Column [all run data] not found

## 3. Базове очищення назв колонок

BehaviourSpace іноді додає пробіли або службові колонки. Тут ми трохи нормалізуємо назви.

In [ ]:
df.columns = [str(c).strip() for c in df.columns]

# Часто у BehaviourSpace є службові колонки, їх можна залишити або прибрати нижче.
print(df.columns.tolist())

['[step]', 'sum [profit] of cafes', 'sum [revenue] of cafes', 'sum [loyalty-cost] of cafes', 'sum [visits] of cafes', 'sum [lost-visits] of customers', 'mean [satisfaction] of customers', '[profit] of one-of cafes with [cafe-id = 0]', '[profit] of one-of cafes with [cafe-id = 1]', '[profit] of one-of cafes with [cafe-id = 2]', '[profit] of one-of cafes with [cafe-id = 3]', '[profit] of one-of cafes with [cafe-id = 4]', '[visits] of one-of cafes with [cafe-id = 0]', '[visits] of one-of cafes with [cafe-id = 1]', '[visits] of one-of cafes with [cafe-id = 2]', '[visits] of one-of cafes with [cafe-id = 3]', '[visits] of one-of cafes with [cafe-id = 4]']


## 4. Налаштування параметрів і KPI

Перевір назви колонок у попередньому виводі й за потреби відредагуй списки.

In [ ]:
parameter_columns = [
    "network-type",
    "number-of-customers",
    "loyalty-effect-multiplier",
    "recommendation-probability",
    "recommendation-boost",
    "mass-market-capacity",
    "premium-capacity",
]

kpi_columns = [
    "sum [profit] of cafes",
    "sum [revenue] of cafes",
    "sum [loyalty-cost] of cafes",
    "sum [visits] of cafes",
    "sum [lost-visits] of customers",
    "mean [satisfaction] of customers",
]

existing_parameters = [c for c in parameter_columns if c in df.columns]
existing_kpis = [c for c in kpi_columns if c in df.columns]

missing_parameters = [c for c in parameter_columns if c not in df.columns]
missing_kpis = [c for c in kpi_columns if c not in df.columns]

print("Existing parameters:", existing_parameters)
print("Missing parameters:", missing_parameters)
print("Existing KPI:", existing_kpis)
print("Missing KPI:", missing_kpis)

Existing parameters: []
Missing parameters: ['network-type', 'number-of-customers', 'loyalty-effect-multiplier', 'recommendation-probability', 'recommendation-boost', 'mass-market-capacity', 'premium-capacity']
Existing KPI: ['sum [profit] of cafes', 'sum [revenue] of cafes', 'sum [loyalty-cost] of cafes', 'sum [visits] of cafes', 'sum [lost-visits] of customers', 'mean [satisfaction] of customers']
Missing KPI: []


## 5. Агрегація повторів експерименту

Якщо BehaviourSpace має кілька повторів для однакових параметрів, краще усереднити KPI.

In [ ]:
if existing_parameters and existing_kpis:
    grouped = (
        df[existing_parameters + existing_kpis]
        .dropna()
        .groupby(existing_parameters, as_index=False)
        .agg({kpi: "mean" for kpi in existing_kpis})
    )
else:
    grouped = df.copy()

print("Rows before aggregation:", len(df))
print("Rows after aggregation:", len(grouped))
grouped.head()

Rows before aggregation: 366
Rows after aggregation: 366


,[step],sum [profit] of cafes,sum [revenue] of cafes,sum [loyalty-cost] of cafes,sum [visits] of cafes,sum [lost-visits] of customers,mean [satisfaction] of customers,[profit] of one-of cafes with [cafe-id = 0],[profit] of one-of cafes with [cafe-id = 1],[profit] of one-of cafes with [cafe-id = 2],[profit] of one-of cafes with [cafe-id = 3],[profit] of one-of cafes with [cafe-id = 4],[visits] of one-of cafes with [cafe-id = 0],[visits] of one-of cafes with [cafe-id = 1],[visits] of one-of cafes with [cafe-id = 2],[visits] of one-of cafes with [cafe-id = 3],[visits] of one-of cafes with [cafe-id = 4]
0,0,0.0,0,0.0,0,0,0.756207,0,0.0,0,0,0.0,0,0,0,0,0
1,1,17448.5,63390,1792.5,1763,0,0.762047,3241,2750.0,2750,4500,4207.5,463,500,500,150,150
2,2,34585.0,125785,3615.0,3495,0,0.767211,6265,5500.0,5500,8905,8415.0,895,1000,1000,300,300
3,3,51854.5,188750,5437.5,5246,0,0.772246,9422,8250.0,8250,13310,12622.5,1346,1500,1500,450,450
4,4,68297.0,251250,7530.0,7001,0,0.777496,12607,11000.0,11000,16860,16830.0,1801,2000,2000,600,600


## 6. Функція аналізу важливості параметрів

Використовуємо Random Forest + permutation importance. Це добре підходить для нелінійної агентної моделі.

In [ ]:
def analyze_parameter_importance(data, parameter_columns, target_column, test_size=0.25, random_state=42):
    data = data[parameter_columns + [target_column]].dropna().copy()

    X = data[parameter_columns]
    y = data[target_column]

    categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_features = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", "passthrough", numeric_features),
        ]
    )

    model = RandomForestRegressor(
        n_estimators=500,
        random_state=random_state,
        min_samples_leaf=3,
        n_jobs=-1,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    metrics = {
        "target": target_column,
        "r2": r2_score(y_test, predictions),
        "mae": mean_absolute_error(y_test, predictions),
    }

    importance = permutation_importance(
        pipeline,
        X_test,
        y_test,
        n_repeats=20,
        random_state=random_state,
        n_jobs=-1,
    )

    importance_df = pd.DataFrame({
        "parameter": X.columns,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    }).sort_values("importance_mean", ascending=False)

    return pipeline, metrics, importance_df

## 7. Аналіз одного KPI

Зміни `target_column`, щоб аналізувати інший результат.

In [ ]:
target_column = "sum [profit] of cafes"

if target_column not in grouped.columns:
    raise ValueError(f"Column not found: {target_column}")

pipeline, metrics, importance_df = analyze_parameter_importance(
    grouped,
    existing_parameters,
    target_column,
)

print(metrics)
importance_df

ValueError: Found array with 0 feature(s) (shape=(274, 0)) while a minimum of 1 is required by RandomForestRegressor.

## 8. Графік важливості параметрів

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(importance_df["parameter"], importance_df["importance_mean"])
plt.xlabel("Permutation importance")
plt.ylabel("Parameter")
plt.title(f"Parameter influence on: {target_column}")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Аналіз усіх KPI одразу

In [ ]:
all_results = []
model_quality = []

for kpi in existing_kpis:
    try:
        _, metrics, imp = analyze_parameter_importance(grouped, existing_parameters, kpi)
        imp = imp.copy()
        imp["target"] = kpi
        all_results.append(imp)
        model_quality.append(metrics)
    except Exception as e:
        print(f"Failed for {kpi}: {e}")

importance_all = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
quality_df = pd.DataFrame(model_quality)

quality_df

## 10. Таблиця важливості параметрів по всіх KPI

In [ ]:
importance_pivot = importance_all.pivot_table(
    index="parameter",
    columns="target",
    values="importance_mean",
    aggfunc="mean"
)

importance_pivot

## 11. Heatmap важливості параметрів

Темніші/більші значення означають сильніший вплив параметра на KPI.

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(importance_pivot.fillna(0), aspect="auto")
plt.colorbar(label="Permutation importance")
plt.xticks(range(len(importance_pivot.columns)), importance_pivot.columns, rotation=45, ha="right")
plt.yticks(range(len(importance_pivot.index)), importance_pivot.index)
plt.title("Parameter importance across KPI")
plt.tight_layout()
plt.show()

## 12. Grouped summary для одного параметра

Корисно, щоб побачити не лише важливість, а й напрям впливу. Наприклад, як змінюється profit для різних `network-type`.

In [ ]:
parameter_to_inspect = "network-type"
target_to_inspect = "sum [profit] of cafes"

summary = grouped.groupby(parameter_to_inspect)[target_to_inspect].agg(["mean", "std", "min", "max", "count"])
summary

## 13. Boxplot для одного параметра

In [ ]:
parameter_to_inspect = "network-type"
target_to_inspect = "sum [profit] of cafes"

labels = []
values = []

for name, group in grouped.groupby(parameter_to_inspect):
    labels.append(str(name))
    values.append(group[target_to_inspect].dropna().values)

plt.figure(figsize=(9, 5))
plt.boxplot(values, labels=labels)
plt.ylabel(target_to_inspect)
plt.xlabel(parameter_to_inspect)
plt.title(f"{target_to_inspect} by {parameter_to_inspect}")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

KeyError: 'network-type'

## 14. Збереження результатів

In [ ]:
importance_df.to_csv("parameter_importance_single_kpi.csv", index=False)
importance_all.to_csv("parameter_importance_all_kpi.csv", index=False)
quality_df.to_csv("model_quality.csv", index=False)
importance_pivot.to_csv("parameter_importance_pivot.csv")

print("Saved files:")
print("- parameter_importance_single_kpi.csv")
print("- parameter_importance_all_kpi.csv")
print("- model_quality.csv")
print("- parameter_importance_pivot.csv")

## Як інтерпретувати

- Якщо параметр має велике `importance_mean`, він сильно впливає на вибраний KPI.
- Якщо `R²` низький, модель Random Forest погано пояснює результат — можливо, треба більше повторів або інші параметри.
- Для статті найкраще показати: heatmap важливості, boxplot для `network-type`, і таблицю важливості для `profit`, `market share`, `lost visits`.